# RAG 三元组与分层指标

前置知识：C5 系统评估与优化、C7.2-C7.6

本节目标：理解 RAG 三元组指标体系，使用 Ragas 对 RAG Pipeline 产出分层评估报告。

## 一、RAG 三元组（RAG Triad）

RAG 系统的评估可以围绕 **Query → Context → Response** 三角形展开，形成三个核心维度：

```
                    Query
                   ╱     ╲
                  ╱       ╲
     Context     ╱         ╲   Answer
     Relevance  ╱           ╲  Relevance
               ╱             ╲
              ╱               ╲
         Context ──────────── Response
                 Faithfulness
               (Groundedness)
```

### 1. 上下文相关性（Context Relevance）

- **检查对象**：检索返回的文档是否与用户问题相关
- **目标阈值**：≥ 0.75
- **低分原因**：检索器返回了太多不相关的文档片段，噪声过大
- **优化方向**：改进 embedding 模型、调整 chunk 策略、优化检索参数

### 2. 忠实度（Faithfulness / Groundedness）

- **检查对象**：生成的答案是否有上下文支撑，是否存在「幻觉」
- **目标阈值**：≥ 0.80
- **低分原因**：LLM 凭空编造信息，没有基于检索到的上下文回答
- **优化方向**：优化 Prompt 模板，明确要求基于上下文回答

### 3. 答案相关性（Answer Relevance）

- **检查对象**：生成的答案是否真正回答了用户的问题
- **目标阈值**：≥ 0.85
- **低分原因**：答非所问，回答偏离了用户意图
- **优化方向**：Query 改写、Prompt 工程优化

## 二、分层指标体系

在三元组的基础上，我们可以按 RAG Pipeline 的阶段将指标分为三层：

### 检索层指标

| 指标 | 含义 | 计算方式 |
|------|------|----------|
| Context Precision | 检索结果中相关文档排在前面的程度 | 相关文档的排名加权得分 |
| Context Recall | 标准答案中的关键信息是否被检索到 | ground_truth 中的信息在 context 中的覆盖率 |

### 生成层指标

| 指标 | 含义 | 计算方式 |
|------|------|----------|
| Faithfulness | 答案中的声明是否能在上下文中找到依据 | 有据声明数 / 总声明数 |
| Answer Relevancy | 答案是否切题回答了问题 | 基于 embedding 相似度计算 |
| Answer Correctness | 答案与标准答案的一致程度 | 综合语义相似度和事实重叠率 |

### 系统层指标

| 指标 | 含义 | 推荐阈值 |
|------|------|----------|
| 端到端延迟 | 从用户提问到收到回答的时间 | < 5 秒 |
| Token 成本 | 单次调用消耗的 token 数 | 视预算而定 |
| 失败率 | 无法生成有效回答的比例 | < 5% |

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

from modelscope import snapshot_download
model_dir = snapshot_download('BAAI/bge-small-zh-v1.5', cache_dir='./models')
print(f"Embedding 模型路径: {model_dir}")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.chat_models import ChatZhipuAI

embedding = HuggingFaceEmbeddings(model_name=model_dir)

api_key = os.environ.get("ZHIPUAI_API_KEY")
llm = ChatZhipuAI(
    model="glm-4-flash",
    temperature=0.0,
    api_key=api_key
)
print("Embedding 和 LLM 初始化完成")

In [ ]:
import re
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

pdf_path = "../3. 索引阶段/data/pumpkin_book.pdf"
persist_dir = "./chroma_db"

def clean_text(text: str) -> str:
    text = re.sub(r'→_→\n.*?←_←', '', text, flags=re.DOTALL)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def build_vectorstore(pdf_path, embedding, persist_directory="./chroma_db"):
    """构建或加载向量库，已有则复用"""
    if os.path.exists(persist_directory) and os.listdir(persist_directory):
        print(f"发现已存在的向量库: {persist_directory}，正在加载...")
        try:
            vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embedding)
            count = vectorstore._collection.count()
            print(f"✅ 加载成功！共 {count} 个文档块")
            return vectorstore
        except Exception as e:
            print(f"⚠️ 加载失败 ({e})，将重新构建...")
    
    print("开始构建向量库...")
    loader = PyMuPDFLoader(pdf_path)
    pdf_pages = loader.load()
    data_pages = pdf_pages[13:-13]
    for page in data_pages:
        page.page_content = clean_text(page.page_content)
    
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    splits = text_splitter.split_documents(data_pages)
    vectorstore = Chroma.from_documents(documents=splits, embedding=embedding, persist_directory=persist_directory)
    print(f"✅ 向量库构建完成并保存至 {persist_directory}，共 {len(splits)} 个文档块")
    return vectorstore

vectorstore = build_vectorstore(pdf_path, embedding, persist_dir)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template(
    "根据以下上下文回答问题。如果上下文中没有相关信息，请说'根据已有资料无法回答'。\n\n"
    "上下文：\n{context}\n\n"
    "问题：{question}\n\n"
    "回答："
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

response = rag_chain.invoke("什么是信息增益？")
print(f"测试回答：{response[:200]}")

In [ ]:
eval_data = [
    {
        "question": "什么是信息增益？",
        "ground_truth": "信息增益是指在得知某个特征的信息后，信息不确定性减少的程度。在决策树中，信息增益越大，说明该特征对分类的贡献越大。"
    },
    {
        "question": "什么是基尼指数？",
        "ground_truth": "基尼指数是度量数据集纯度的一种指标，反映了从数据集中随机抽取两个样本，其类别标记不一致的概率。基尼指数越小，数据集纯度越高。"
    },
    {
        "question": "过拟合和欠拟合的区别是什么？",
        "ground_truth": "过拟合是模型在训练集上表现好但在测试集上表现差，学到了训练数据中的噪声。欠拟合是模型在训练集和测试集上都表现不好，没有学到数据的基本规律。"
    },
    {
        "question": "什么是支持向量机？",
        "ground_truth": "支持向量机是一种二分类模型，通过在特征空间中找到一个最优超平面来实现分类，使得两类样本到超平面的间隔最大化。"
    },
    {
        "question": "朴素贝叶斯分类器的基本原理是什么？",
        "ground_truth": "朴素贝叶斯分类器基于贝叶斯定理，假设各特征之间条件独立，通过计算后验概率来进行分类，选择后验概率最大的类别作为预测结果。"
    },
]
print(f"准备了 {len(eval_data)} 条评估数据")

In [ ]:
from tqdm import tqdm

results = []
for item in tqdm(eval_data, desc="生成回答"):
    docs = retriever.invoke(item["question"])
    answer = rag_chain.invoke(item["question"])
    results.append({
        "question": item["question"],
        "answer": answer,
        "contexts": [doc.page_content for doc in docs],
        "ground_truth": item["ground_truth"]
    })

print(f"已生成 {len(results)} 条回答")
print(f"\n示例：\n问题：{results[0]['question']}\n回答：{results[0]['answer'][:200]}")

In [ ]:
# !pip install ragas datasets

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset

eval_dataset = Dataset.from_dict({
    "question": [r["question"] for r in results],
    "answer": [r["answer"] for r in results],
    "contexts": [r["contexts"] for r in results],
    "ground_truth": [r["ground_truth"] for r in results],
})

ragas_llm = LangchainLLMWrapper(llm)
ragas_emb = LangchainEmbeddingsWrapper(embedding)

eval_result = evaluate(
    dataset=eval_dataset,
    metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
    llm=ragas_llm,
    embeddings=ragas_emb,
)

print("=" * 50)
print("分层评估报告")
print("=" * 50)
for metric, score in eval_result.items():
    if isinstance(score, float):
        print(f"  {metric}: {score:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

metric_names = [k for k, v in eval_result.items() if isinstance(v, float)]
scores = [v for v in eval_result.values() if isinstance(v, float)]

angles = np.linspace(0, 2 * np.pi, len(metric_names), endpoint=False).tolist()
scores_plot = scores + [scores[0]]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.fill(angles, scores_plot, alpha=0.25, color='steelblue')
ax.plot(angles, scores_plot, 'o-', linewidth=2, color='steelblue')
ax.set_thetagrids(np.degrees(angles[:-1]), metric_names)
ax.set_ylim(0, 1)
ax.set_title("RAG 分层评估雷达图", pad=20, fontsize=14)
plt.tight_layout()
plt.savefig("./figures/ragas_radar.png", dpi=150, bbox_inches='tight')
plt.show()

## 三、评估结果解读

### 怎么看这些分数？

| 指标 | 得分区间 | 含义 |
|------|---------|------|
| Context Precision | ≥ 0.80 优 / 0.60-0.80 可 / < 0.60 差 | 检索结果是否精准 |
| Context Recall | ≥ 0.75 优 / 0.50-0.75 可 / < 0.50 差 | 关键信息是否被检索到 |
| Faithfulness | ≥ 0.80 优 / 0.60-0.80 可 / < 0.60 差 | 答案是否有据可查 |
| Answer Relevancy | ≥ 0.85 优 / 0.70-0.85 可 / < 0.70 差 | 答案是否切题 |

### 常见问题诊断

- **Context Precision 低**：检索返回了太多无关文档 → 考虑调整 chunk_size 或换用更好的 embedding
- **Context Recall 低**：关键信息没有被检索到 → 考虑增大 k 值或优化文档切分策略
- **Faithfulness 低**：模型在编造内容 → 检查 prompt 是否明确要求基于上下文回答
- **Answer Relevancy 低**：答非所问 → 检查 prompt 模板或考虑 query 改写

In [ ]:
try:
    df = eval_result.to_pandas()
    print(df.to_string(index=False))
except AttributeError:
    print("逐条得分:")
    for i, r in enumerate(results):
        print(f"  [{i+1}] {r['question'][:20]}...")
    print("\n汇总:", eval_result)

## 四、小结

本节介绍了 RAG 三元组和分层指标体系，并使用 Ragas 对 RAG Pipeline 进行了分层评估。

### 实践建议

1. **先定阈值再优化**：为每个指标设定目标阈值，低于阈值的重点优化
2. **检索优先**：如果 Context Precision/Recall 低，先优化检索再调 Prompt
3. **定期跑评估**：每次改动后跑一轮，避免改好了 A 却破坏了 B
4. **指标不是万能的**：数值高不代表用户满意，数值低也不代表一定有问题，需结合人工抽检

### 下一步

在下一节中，我们将学习如何手写 LLM-as-Judge，实现更灵活的自定义评估。

### 参考文献

- [Ragas 官方文档](https://docs.ragas.io/)
- [RAG Evaluation: 2026 Metrics and Benchmarks](https://labelyourdata.com/articles/llm-fine-tuning/rag-evaluation)